# 🎙️ VoiceForge AI — Custom RVC v2 Voice Profile Training
This Google Colab notebook fine-tunes a personalized neural voice model from your 5–10 minute voice dataset using Retrieval-based Voice Conversion (RVC v2).

**Hardware:** Ensure GPU acceleration is enabled: `Runtime` -> `Change runtime type` -> `T4 GPU`.

In [ ]:
# Step 1: Verify GPU & Clone RVC v2 Repository
!nvidia-smi
!git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git /content/RVC
%cd /content/RVC
!pip install -r requirements.txt
!pip install torchcrepe fairseq praat-parselmouth

In [ ]:
# Step 2: Download Foundation Pretrained Checkpoints
!mkdir -p /content/RVC/assets/pretrained_v2 /content/RVC/assets/hubert /content/RVC/assets/rmvpe
# Download HuBERT base
!wget -q -O /content/RVC/assets/hubert/hubert_base.pt https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt
# Download RMVPE pitch estimator
!wget -q -O /content/RVC/assets/rmvpe/rmvpe.pt https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt
# Download v2 40k base weights
!wget -q -O /content/RVC/assets/pretrained_v2/f0G40k.pth https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth
!wget -q -O /content/RVC/assets/pretrained_v2/f0D40k.pth https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth
print('✓ Foundation models downloaded!')

In [ ]:
# Step 3: Upload and Extract Your Dataset
from google.colab import files
import os
import zipfile

print('Please upload user_voice_dataset.zip (exported from VoiceForge AI):')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

dataset_dir = '/content/RVC/dataset_user'
os.makedirs(dataset_dir, exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(dataset_dir)

print(f'✓ Extracted {len(os.listdir(dataset_dir))} audio files into {dataset_dir}!')

In [ ]:
# Step 4: Feature Extraction & Dataset Preprocessing
MODEL_NAME = 'my_custom_voice'

# 1. Slice and preprocess
!python infer/modules/train/preprocess.py /content/RVC/dataset_user 40000 2 /content/RVC/logs/{MODEL_NAME} False 3.0

# 2. Extract pitch (f0) using RMVPE and HuBERT 16k content representations
!python infer/modules/train/extract/extract_f0_rmvpe.py 1 0 0 /content/RVC/logs/{MODEL_NAME} True
!python infer/modules/train/extract_feature_print.py cuda:0 1 0 0 /content/RVC/logs/{MODEL_NAME} v2
print('✓ Feature extraction complete!')

In [ ]:
# Step 5: Fine-Tune Voice Model
# Total epochs: 180 (usually takes ~12-15 minutes on a T4 GPU)
!python infer/modules/train/train.py -e {MODEL_NAME} -sr 40k -f0 1 -bs 8 -g 0 -te 180 -se 20 -pg assets/pretrained_v2/f0G40k.pth -pd assets/pretrained_v2/f0D40k.pth -l 0 -c 0 -sw 0 -v v2

# Step 6: Build Feature Retrieval Index
!python infer/modules/train/train_index.py /content/RVC/logs/{MODEL_NAME} v2
print('✓ Training and index construction complete!')

In [ ]:
# Step 7: Export and Download Model Files
import shutil
import glob
from google.colab import files

export_dir = f'/content/export_{MODEL_NAME}'
os.makedirs(export_dir, exist_ok=True)

# Copy model .pth
pth_files = glob.glob(f'/content/RVC/assets/weights/{MODEL_NAME}.pth')
if pth_files:
    shutil.copy(pth_files[0], export_dir)

# Copy .index file
index_files = glob.glob(f'/content/RVC/logs/{MODEL_NAME}/added_*.index')
for f in index_files:
    shutil.copy(f, export_dir)

# Zip for clean download
zip_out = f'/content/{MODEL_NAME}_trained_model.zip'
shutil.make_archive(zip_out.replace('.zip', ''), 'zip', export_dir)
print(f'Downloading {zip_out}...')
files.download(zip_out)
print('✓ Done! Extract the .pth and .index files into backend/models/ on your computer.')